##NBA Multi-Dimensional Analytics & Scouting Engine

A data pipeline and analytics framework for integrating NBA tracking, synergy, box score, and shot data into a unified player-level dataset.

**Key Features**

Multi-source data integration (tracking, synergy, box score, shot locations)
Centralized master dataset with 500+ engineered features
Regular season vs playoff comparison for contextual analysis
Modular ingestion pipeline with caching (Parquet-based)
Built for downstream modeling (e.g., RAPM, salary prediction)

# Import Packages

In [40]:
pip install shap

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.2/559.2 kB 3.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 17.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.1/31.1 MB 32.7 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [shap]4/5 [shap]]te]
Note: you may need to restart the kernel to use updated packages.


In [125]:
# Standard library
import gc
import itertools
import pickle
import time
import warnings
from functools import reduce
from random import sample
from unidecode import unidecode
import re, requests
from pathlib import Path
import sklearn
import shap
import joblib
import json

# Core scientific stack
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from matplotlib.offsetbox import AnnotationBbox, OffsetImage
from matplotlib.patches import Rectangle
from matplotlib.ticker import FormatStrFormatter
import seaborn as sns
import lightgbm as lgb
from sklearn.metrics import r2_score, mean_absolute_error

import holoviews as hv
import hvplot.pandas  # noqa: F401 (registers hvplot accessor)
hv.extension("bokeh")

# Stats
from scipy import stats

# NBA API
from nba_api.stats.static.teams import get_teams
from nba_api.stats.static.players import get_players, get_active_players
from nba_api.stats.endpoints import leaguegamefinder, leaguedashplayerstats

import importlib
import nba_viz_utils
importlib.reload(nba_viz_utils)


from nba_viz_utils import *

In [2]:
pwd

'/Users/siddharthravindran/Documents/nba_data_vis'

# Upload Master

## Load Master Data Frame

In [3]:
df_master = pd.read_parquet(
  "data/df_master_current.parquet"
).set_index(INDEX_COLS).sort_index()

## Master Integrity Check

In [4]:
check_master_integrity(df_master)

📊 Integrity Check for df_master:
  - Shape: (8373, 730)
  - Total Players: 1556
  - Seasons Present: ['2015-16', '2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25', '2025-26']
  - ✅ No duplicate indices found.
  - ✅ All columns contain at least some data.


# Missing Data Check

In [4]:
nan_by_season = df_master.groupby(level='SEASON').apply(lambda g: g.isna().mean())
flip = nan_by_season.columns[(nan_by_season.max() > 0.95) & (nan_by_season.min() < 0.5)]
nan_by_season[flip].round(2)

""
SEASON
2015-16
2016-17
2017-18
2018-19
2019-20
2020-21
2021-22
2022-23
2023-24


# Transpose Regular Season & Playoffs

In [5]:
model = build_model_table(df_master)

In [6]:
model

,PLAYER_ID,SEASON,PLAYER_NAME,TEAM_ABBREVIATION,TEAM_ID,GP_rs,W_rs,L_rs,W_PCT_rs,MIN_rs,...,CLUTCH_TS_PCT_po,CLUTCH_USG_PCT_po,CLUTCH_E_PACE_po,CLUTCH_PACE_po,CLUTCH_PACE_PER40_po,CLUTCH_PIE_po,CLUTCH_POSS_po,CLUTCH_FGM_PG_po,CLUTCH_FGA_PG_po,made_playoffs
0,201950,2019-20,Jrue Holiday,NOP,1610612740,61,26,35,0.426,34.7,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,1627759,2019-20,Jaylen Brown,BOS,1610612738,57,38,19,0.667,33.9,...,0.444,0.186,94.69,94.03,78.36,0.118,100.0,0.5,1.5,1
2,203935,2019-20,Marcus Smart,BOS,1610612738,60,39,21,0.650,32.0,...,0.429,0.116,93.94,92.74,77.28,0.063,96.0,0.2,0.6,1
3,1628970,2019-20,Miles Bridges,CHA,1610612766,65,23,42,0.354,30.7,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,202710,2019-20,Jimmy Butler III,MIA,1610612748,58,38,20,0.655,33.8,...,0.760,0.342,88.07,87.53,72.94,0.341,96.0,0.9,1.5,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5963,204066,2018-19,John Holland,CLE,1610612739,1,0,1,0.000,0.7,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
5964,1629122,2019-20,J.P. Macura,CLE,1610612739,1,1,0,1.000,0.6,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
5965,1630701,2022-23,Michael Foster Jr.,PHI,1610612755,1,1,0,1.000,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
5966,1628382,2023-24,Justin Jackson,MIN,1610612750,2,2,0,1.000,0.4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0


In [7]:
m = model.reset_index()
m['PLAYER_NAME'] = m['PLAYER_NAME'].apply(norm_name)
print("dupe player-seasons:", m.duplicated(subset=['PLAYER_ID','SEASON']).sum())

dupe player-seasons: 0


In [10]:
# m.to_parquet("data/transposed_df_master.parquet")

model = pd.read_parquet("data/transposed_df_master.parquet")

In [11]:
model

,index,PLAYER_ID,SEASON,PLAYER_NAME,TEAM_ABBREVIATION,TEAM_ID,GP_rs,W_rs,L_rs,W_PCT_rs,...,CLUTCH_TS_PCT_po,CLUTCH_USG_PCT_po,CLUTCH_E_PACE_po,CLUTCH_PACE_po,CLUTCH_PACE_PER40_po,CLUTCH_PIE_po,CLUTCH_POSS_po,CLUTCH_FGM_PG_po,CLUTCH_FGA_PG_po,made_playoffs
0,0,201950,2019-20,jrue holiday,NOP,1610612740,61,26,35,0.426,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,1,1627759,2019-20,jaylen brown,BOS,1610612738,57,38,19,0.667,...,0.444,0.186,94.69,94.03,78.36,0.118,100.0,0.5,1.5,1
2,2,203935,2019-20,marcus smart,BOS,1610612738,60,39,21,0.650,...,0.429,0.116,93.94,92.74,77.28,0.063,96.0,0.2,0.6,1
3,3,1628970,2019-20,miles bridges,CHA,1610612766,65,23,42,0.354,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,4,202710,2019-20,jimmy butler,MIA,1610612748,58,38,20,0.655,...,0.760,0.342,88.07,87.53,72.94,0.341,96.0,0.9,1.5,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5963,5963,204066,2018-19,john holland,CLE,1610612739,1,0,1,0.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
5964,5964,1629122,2019-20,jp macura,CLE,1610612739,1,1,0,1.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
5965,5965,1630701,2022-23,michael foster,PHI,1610612755,1,1,0,1.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
5966,5966,1628382,2023-24,justin jackson,MIN,1610612750,2,2,0,1.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0


# Salaries

In [ ]:
all_urls = harvest_player_urls()        # all 11 seasons → ~1,800 unique players
salaries  = scrape_all(all_urls)
print(salaries.shape); salaries.head()

  2015-16: 487 players (total unique 487)
  2016-17: 494 players (total unique 586)
  2017-18: 549 players (total unique 722)
  2018-19: 537 players (total unique 830)
  2019-20: 538 players (total unique 949)
  2020-21: 547 players (total unique 1043)
  2021-22: 613 players (total unique 1170)
  2022-23: 548 players (total unique 1255)
  2023-24: 578 players (total unique 1355)
  2024-25: 575 players (total unique 1459)
  2025-26: 588 players (total unique 1562)
  50/1562
  100/1562
  150/1562
  200/1562
  250/1562
  300/1562
  350/1562
  400/1562
  450/1562
  500/1562
  550/1562
  600/1562
  650/1562
  700/1562
  750/1562
  800/1562
  850/1562
  900/1562
  950/1562
  1000/1562
  1050/1562
  1100/1562
  1150/1562
  1200/1562
  1250/1562
  1300/1562
  1350/1562
  1400/1562
  1450/1562
  1500/1562
  1550/1562
(7701, 4)


,SEASON,Salary,Player,name_key
0,2016-17,5994764,Álex Abrines,alex abrines
1,2017-18,5725000,Álex Abrines,alex abrines
2,2018-19,3575183,Álex Abrines,alex abrines
3,2020-21,2582160,Precious Achiuwa,precious achiuwa
4,2021-22,2711280,Precious Achiuwa,precious achiuwa


In [6]:
# salaries.to_csv(DATA_DIR / "bbref_salaries_2015-2025.csv", index=False)

salaries = pd.read_csv("data/bbref_salaries_2015-2025.csv")
salaries = salaries[salaries['SEASON'].between('2015-16', '2024-25')].reset_index(drop=True)

In [7]:
salaries_25_26 = pd.read_csv("data/contracts.csv")

In [ ]:
col_2526 = next(c for c in salaries_25_26.columns if '2025' in str(c))   # column NAME → separate var

def to_int_salary(s):
    digits = re.sub(r'[^0-9]', '', str(s).split('.')[0])   # split('.') guards against a float '45780966.0'
    return int(digits) if digits else 0

current = salaries_25_26[['Player', col_2526]].dropna().copy()
current['Salary']   = current[col_2526].map(to_int_salary)
current = current[current['Salary'] > 0]
current['SEASON']   = '2025-26'
current['name_key'] = current['Player'].apply(norm_name)
current = current[['SEASON', 'Salary', 'Player', 'name_key']]
print(current.shape); current.head()

(529, 4)


,SEASON,Salary,Player,name_key
0,2025-26,59606817,Stephen Curry,stephen curry
1,2025-26,55224526,Joel Embiid,joel embiid
2,2025-26,55224526,Nikola Jokić,nikola jokic
3,2025-26,54708609,Kevin Durant,kevin durant
4,2025-26,54126450,Jayson Tatum,jayson tatum


In [32]:
salaries = pd.concat([salaries, current[['SEASON', 'Salary', 'Player', 'name_key']]], ignore_index=True)
salaries = salaries.drop_duplicates(['name_key', 'SEASON'])

In [12]:
# salaries.to_csv("data/bbref_salaries_2015-2026.csv", index=False)
salaries = pd.read_csv("data/bbref_salaries_2015-2026.csv")

# Build Model Data Frame

In [13]:
model = attach_salaries(model, salaries)    

In [14]:
# did the scrape actually fix the ~14% hole?
real = model[(model['GP_rs'] >= 20) & (model['MIN_rs'] >= 10)]
print(f"coverage (real-minute players): {real['Salary'].notna().mean():.1%}  "
      f"| missing rows: {real['Salary'].isna().sum()}")

coverage (real-minute players): 97.7%  | missing rows: 100


In [ ]:
# draft / experience — model-layer attributes, merged on PLAYER_ID
model['PLAYER_ID'] = model['PLAYER_ID'].astype(str)
bio = get_draft_table(SEASONS)
bio['PLAYER_ID'] = bio['PLAYER_ID'].astype(str)
model = (model.drop(columns=['DRAFT_POSITION', 'DRAFT_YR'], errors='ignore')   # idempotent
              .merge(bio, on='PLAYER_ID', how='left'))

model['IS_UNDRAFTED']     = model['DRAFT_POSITION'].isna().astype(int)
model['EXPERIENCE']       = (model['SEASON'].str[:4].astype(int) - model['DRAFT_YR']).clip(lower=0)
model['MAX_PCT_ELIGIBLE'] = model['EXPERIENCE'].map(
    lambda e: np.nan if pd.isna(e) else (0.25 if e <= 6 else 0.30 if e <= 9 else 0.35))

# target + filter LAST (so `model` stays the full scoreable population)
model['pct_cap'] = model['Salary'] / model['SEASON'].map(SALARY_CAP)

In [100]:
for nm in ['giannis', 'embiid']:
    sub = model[model['PLAYER_NAME'].str.contains(nm, case=False, na=False)]
    print(sub[['PLAYER_NAME','SEASON','ALLNBA_PRIOR3','ALLNBA_PRIOR_EVER']]
          .sort_values('SEASON').to_string(index=False), "\n")

# and the source table — how many selections did the parse actually capture for each?
print("Giannis in parsed All-NBA table:")
print(an[an['name_key'].str.contains('antetok', case=False, na=False)].sort_values('SEASON').to_string(index=False))
print("\nEmbiid in parsed All-NBA table:")
print(an[an['name_key'].str.contains('embiid', case=False, na=False)].sort_values('SEASON').to_string(index=False))

          PLAYER_NAME  SEASON  ALLNBA_PRIOR3  ALLNBA_PRIOR_EVER
giannis antetokounmpo 2015-16              0                  0
giannis antetokounmpo 2016-17              0                  0
 georgios papagiannis 2016-17              0                  0
giannis antetokounmpo 2017-18              1                  1
 georgios papagiannis 2017-18              0                  0
giannis antetokounmpo 2018-19              2                  2
giannis antetokounmpo 2019-20              3                  3
giannis antetokounmpo 2020-21              3                  4
giannis antetokounmpo 2021-22              3                  5
giannis antetokounmpo 2022-23              3                  6
giannis antetokounmpo 2023-24              3                  7
giannis antetokounmpo 2024-25              3                  8
giannis antetokounmpo 2025-26              3                  9 

PLAYER_NAME  SEASON  ALLNBA_PRIOR3  ALLNBA_PRIOR_EVER
joel embiid 2016-17              0              

In [ ]:
def fetch_all_nba_raw(url="https://www.basketball-reference.com/awards/all_league.html"):
    r = requests.get(url, headers=HDR, timeout=30)
    print("status:", r.status_code)
    soup = BeautifulSoup(r.content, "html.parser")
    for c in soup.find_all(string=lambda t: isinstance(t, Comment)):     # BBRef comment-hides tables
        if 'all_league' in c.lower() and '<table' in c.lower():
            soup.append(BeautifulSoup(c, "html.parser"))
    tbl = soup.find("table", id=re.compile("all.?league", re.I)) or soup.find("table")
    df = pd.read_html(StringIO(str(tbl)))[0]
    print("shape:", df.shape, "| cols:", list(df.columns))
    return df

allnba_raw = fetch_all_nba_raw()
allnba_raw.head(12)

In [59]:
# --- parse the raw All-NBA table → long (name_key, SEASON, team_level) ---
player_cols = ['Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8']

an = allnba_raw[allnba_raw['Lg'] == 'NBA'].copy()          # drops blank rows + any ABA history
an = an.melt(id_vars=['Season', 'Tm'], value_vars=player_cols, value_name='player_raw')
an = an.dropna(subset=['player_raw'])

# strip the trailing position token(s): "Giannis Antetokounmpo F" / "... G-F" → name
an['player'] = an['player_raw'].str.replace(r'\s+[CFG](-[CFG])?$', '', regex=True)
an['name_key'] = an['player'].apply(norm_name)
an = an.rename(columns={'Season': 'SEASON'})
an = an[an['SEASON'].isin(SEASONS)][['name_key', 'SEASON', 'Tm']]
print("All-NBA selections parsed:", an.shape)
print(an.head())

# --- credential features: All-NBA in PRIOR seasons (causally clean + matches supermax rule) ---
an['yr'] = an['SEASON'].str[:4].astype(int)
picks = an.groupby(['name_key', 'yr']).size().rename('n').reset_index()   # 1 row per player-season selected

def prior_counts(name_key, year):
    p = picks[picks['name_key'] == name_key]
    last3 = int(p[(p['yr'] < year) & (p['yr'] >= year - 3)]['n'].sum())
    ever  = int(p[p['yr'] < year]['n'].sum())
    return last3, ever

model['name_key'] = model['PLAYER_NAME'].apply(norm_name)
yrs = model['SEASON'].str[:4].astype(int)
counts = [prior_counts(nk, y) for nk, y in zip(model['name_key'], yrs)]
model['ALLNBA_PRIOR3']     = [c[0] for c in counts]      # selections in trailing 3 seasons
model['ALLNBA_PRIOR_EVER'] = [c[1] for c in counts]      # career selections before this season
model = model.drop(columns='name_key')

print("\nplayers with ALLNBA_PRIOR3 > 0:", (model['ALLNBA_PRIOR3'] > 0).sum())
print(model[model['ALLNBA_PRIOR3'] > 0][['PLAYER_NAME','SEASON','ALLNBA_PRIOR3','ALLNBA_PRIOR_EVER','pct_cap']]
      .sort_values('ALLNBA_PRIOR_EVER', ascending=False).head(10))

All-NBA selections parsed: (165, 3)
       name_key   SEASON   Tm
0  nikola jokic  2025-26  1st
1  jaylen brown  2025-26  2nd
2   jalen duren  2025-26  3rd
3  nikola jokic  2024-25  1st
4   evan mobley  2024-25  2nd

players with ALLNBA_PRIOR3 > 0: 243
                PLAYER_NAME   SEASON  ALLNBA_PRIOR3  ALLNBA_PRIOR_EVER  \
140            lebron james  2025-26              3                 10   
62             lebron james  2024-25              3                  9   
756           stephen curry  2025-26              3                  9   
1174  giannis antetokounmpo  2025-26              3                  9   
628   giannis antetokounmpo  2024-25              3                  8   
361           stephen curry  2024-25              3                  8   
260            lebron james  2023-24              3                  8   
1203  giannis antetokounmpo  2023-24              3                  7   
371            nikola jokic  2025-26              3                  7   
552    

In [101]:
# weight selection quality: 1st = 3, 2nd = 2, 3rd = 1
TM_WEIGHT = {'1st': 3, '2nd': 2, '3rd': 1}
an['weight'] = an['Tm'].map(TM_WEIGHT)

aw = an.dropna(subset=['weight']).copy()
aw['yr'] = aw['SEASON'].str[:4].astype(int)

# weighted credential: prior-ever and prior-3 in WEIGHTED points, not raw counts
def prior_weighted(name_key, year):
    p = aw[aw['name_key'] == name_key]
    ever  = int(p[p['yr'] <  year]['weight'].sum())
    last3 = int(p[(p['yr'] < year) & (p['yr'] >= year - 3)]['weight'].sum())
    return ever, last3

model['name_key'] = model['PLAYER_NAME'].apply(norm_name)
yrs = model['SEASON'].str[:4].astype(int)
wc = [prior_weighted(nk, y) for nk, y in zip(model['name_key'], yrs)]
model['ALLNBA_WT_EVER'] = [c[0] for c in wc]
model['ALLNBA_WT3']     = [c[1] for c in wc]
model = model.drop(columns='name_key')

# check the separation you care about
print(model[model['PLAYER_NAME'].str.contains('giannis|embiid', case=False, na=False)]
      [['PLAYER_NAME','SEASON','ALLNBA_WT_EVER','ALLNBA_WT3']]
      .query("SEASON == '2025-26'").to_string(index=False))

          PLAYER_NAME  SEASON  ALLNBA_WT_EVER  ALLNBA_WT3
          joel embiid 2025-26              11           3
giannis antetokounmpo 2025-26              25           9


In [103]:
print("seasons in model:", sorted(model['SEASON'].unique()))
print("PLAYER_ID dtype:", model['PLAYER_ID'].dtype, "| nulls:", model['PLAYER_ID'].isna().sum())
# a known multi-season guy should show one row per season, same ID
print(model[model['PLAYER_NAME'].str.contains('giannis antetokounmpo', case=False, na=False)]
      [['PLAYER_ID','SEASON','MIN_rs','FGM_rs']].sort_values('SEASON').to_string(index=False))

seasons in model: ['2015-16', '2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25', '2025-26']
PLAYER_ID dtype: object | nulls: 0
PLAYER_ID  SEASON  MIN_rs  FGM_rs
   203507 2015-16    35.3     6.4
   203507 2016-17    35.6     8.2
   203507 2017-18    36.7     9.9
   203507 2018-19    32.8    10.0
   203507 2019-20    30.4    10.9
   203507 2020-21    33.0    10.3
   203507 2021-22    32.9    10.3
   203507 2022-23    32.1    11.2
   203507 2023-24    35.2    11.5
   203507 2024-25    34.2    11.8
   203507 2025-26    28.9    10.4


In [84]:
# availability — explicit, so the model stops confusing "hurt" with "declined"
model['GP_rs']  = model['GP_rs'].fillna(0)
model['AVAILABILITY'] = model['GP_rs'] / 82.0                 # share of season played
model['TOTAL_MIN_rs'] = model['MIN_rs'] * model['GP_rs']      # volume: high only if good AND available

In [105]:
# the spine to lag — production identity, not all 1000 cols
SPINE = ['MIN_rs','FGM_rs','FTM_rs','PIE_rs','USG_PCT_rs','FRONT_CT_TOUCHES_rs',
         'POST_TOUCHES_rs','CLOSESTDEF_4_6_FGM_rs','GP_rs','TOTAL_MIN_rs',
         'ALLNBA_WT_EVER']
SPINE = [c for c in SPINE if c in model.columns]

# integer season key so shift respects chronological order
model['_yr'] = model['SEASON'].str[:4].astype(int)
model = model.sort_values(['PLAYER_ID','_yr'])

# lag-1 and lag-2: the player's prior-season and two-seasons-ago production
for k in (1, 2):
    lagged = model.groupby('PLAYER_ID')[SPINE].shift(k)
    # only valid if the prior row is the IMMEDIATELY preceding season (no gap)
    prev_yr = model.groupby('PLAYER_ID')['_yr'].shift(k)
    gap_ok  = (model['_yr'] - prev_yr) == k
    lagged  = lagged.where(gap_ok)                      # NaN out skipped/missing seasons
    lagged.columns = [f"{c}_lag{k}" for c in SPINE]
    model = pd.concat([model, lagged], axis=1)

model = model.drop(columns='_yr')

# sanity: Giannis should now carry last year's healthy minutes alongside this year's
print(model[model['PLAYER_NAME'].str.contains('giannis antetokounmpo', case=False, na=False)]
      [['SEASON','MIN_rs','MIN_rs_lag1','GP_rs','GP_rs_lag1']].sort_values('SEASON').to_string(index=False))

 SEASON  MIN_rs  MIN_rs_lag1  MIN_rs_lag1  GP_rs  GP_rs_lag1  GP_rs_lag1
2015-16    35.3          NaN          NaN     80         NaN         NaN
2016-17    35.6         35.3         35.3     80        80.0        80.0
2017-18    36.7         35.6         35.6     75        80.0        80.0
2018-19    32.8         36.7         36.7     72        75.0        75.0
2019-20    30.4         32.8         32.8     63        72.0        72.0
2020-21    33.0         30.4         30.4     61        63.0        63.0
2021-22    32.9         33.0         33.0     67        61.0        61.0
2022-23    32.1         32.9         32.9     63        67.0        67.0
2023-24    35.2         32.1         32.1     73        63.0        63.0
2024-25    34.2         35.2         35.2     67        73.0        73.0
2025-26    28.9         34.2         34.2     36        67.0        67.0


In [120]:
model = model.drop(columns=[c for c in model.columns if '_lag' in c], errors='ignore')  # top of lag cell
model = model.loc[:, ~model.columns.duplicated()]                                         # belt-and-suspenders

In [110]:
model = model.loc[:, ~model.columns.duplicated()]

mdf = model[model['Salary'].notna() & (model['GP_rs'] >= 20) & (model['MIN_rs'] >= 10)].copy()
mdf['pct_cap'] = mdf['Salary'] / mdf['SEASON'].map(SALARY_CAP)
mdf = mdf[mdf['pct_cap'].notna()].copy()

# does the credential separate the pay tiers the way we expect?
# print(mdf.groupby(mdf['ALLNBA_PRIOR3'].clip(upper=3))['pct_cap'].agg(['mean','count']))
print("mdf:", mdf.shape)
print(mdf['pct_cap'].describe())

mdf: (4228, 1501)
count    4228.000000
mean        0.081441
std         0.084130
min         0.000047
25%         0.018757
50%         0.046164
75%         0.116588
max         0.407253
Name: pct_cap, dtype: float64


In [107]:
# 3) CONFIRM they're present before training
print([c for c in ['AVAILABILITY','TOTAL_MIN_rs'] if c in mdf.columns], "→ should list both")

['AVAILABILITY', 'TOTAL_MIN_rs'] → should list both


In [109]:
print([c for c in model.columns if c.endswith('_lag1')][:15])

['MIN_rs_lag1', 'FGM_rs_lag1', 'FTM_rs_lag1', 'PIE_rs_lag1', 'USG_PCT_rs_lag1', 'FRONT_CT_TOUCHES_rs_lag1', 'POST_TOUCHES_rs_lag1', 'CLOSESTDEF_4_6_FGM_rs_lag1', 'GP_rs_lag1', 'TOTAL_MIN_rs_lag1', 'ALLNBA_WT_EVER_lag1', 'MIN_rs_lag1', 'FGM_rs_lag1', 'FTM_rs_lag1', 'PIE_rs_lag1']


In [20]:
mdf.to_parquet("data/transposed_df_master_with_bio.parquet")
# model = pd.read_parquet("data/transposed_df_master_with_bio.parquet")

# Feature Selection

In [112]:

# 4) re-prefilter (regenerates keep WITH the new cols) then retrain
keep, report = prefilter_features(mdf); print(report)
feats = keep + [c for c in ['AVAILABILITY','TOTAL_MIN_rs'] if c not in keep]
print("availability in feats?", 'AVAILABILITY' in feats, '| total_min in feats?', 'TOTAL_MIN_rs' in feats)

{'start': 1487, 'const': 1, 'empty': 2, 'collinear': 449, 'keep': 1035}
availability in feats? True | total_min in feats? True


In [133]:
feats.remove('DRAFT_YR')   # not a real feature, just a proxy for experience

In [134]:
print("draft year in feats?", 'DRAFT_YR' in feats)

draft year in feats? False


In [135]:

train = mdf[mdf['SEASON'] <= '2021-22']
valid = mdf[mdf['SEASON'] == '2022-23']
test  = mdf[mdf['SEASON'] >= '2023-24']

m_lgb = lgb.LGBMRegressor(
    n_estimators=3000, learning_rate=0.02, num_leaves=31, min_child_samples=30,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.6, reg_lambda=1.0,
    random_state=42, n_jobs=-1)
m_lgb.fit(train[feats], train['pct_cap'],
          eval_set=[(valid[feats], valid['pct_cap'])],
          callbacks=[lgb.early_stopping(150), lgb.log_evaluation(0)])

pred = m_lgb.predict(test[feats])
mae  = mean_absolute_error(test['pct_cap'], pred)
print(f"test R2: {r2_score(test['pct_cap'], pred):.3f}  MAE: {mae:.4f}  (~${mae*SALARY_CAP['2025-26']:,.0f})")

imp = pd.Series(m_lgb.booster_.feature_importance(importance_type='gain'),
                index=feats).sort_values(ascending=False)
print(f"\n>0 gain: {(imp>0).sum()}/{len(feats)}")
print(imp.head(15))

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.025922 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 144567
[LightGBM] [Info] Number of data points in the train set: 2673, number of used features: 1036
[LightGBM] [Info] Start training from score 0.079533
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[488]	valid_0's l2: 0.00123019
test R2: 0.845  MAE: 0.0241  (~$3,733,127)

>0 gain: 979/1036
EXPERIENCE                  69.029709
MIN_rs                      44.010718
MIN_rs_lag1                 27.761961
FGM_rs_lag1                 23.135948
FGM_rs                      15.699504
FRONT_CT_TOUCHES_rs         11.055699
FGM_rs_lag2                 10.394628
MAX_PCT_ELIGIBLE             8.596923
MIN_rs_lag2                  7.346694
ALLNBA_PRIOR_EVER            5.486394
FTM_rs_lag1                  5.306457
DRAFT_POSITION               5.069300
PIE_rs_la

In [136]:
expl   = shap.TreeExplainer(m_lgb)
Xall   = mdf[feats]
sv     = expl.shap_values(Xall)                       # (n_rows, n_feats)

# --- global: mean |SHAP| — the honest importance ranking ---
glob = (pd.Series(np.abs(sv).mean(0), index=feats)
          .sort_values(ascending=False))
print("top 50 by mean |SHAP|:"); print(glob.head(50))

top 50 by mean |SHAP|:
EXPERIENCE                         0.024579
MIN_rs                             0.010397
MIN_rs_lag1                        0.006075
DRAFT_POSITION                     0.003988
MAX_PCT_ELIGIBLE                   0.003847
MIN_rs_lag2                        0.003752
FRONT_CT_TOUCHES_rs                0.003691
FGM_rs_lag1                        0.003268
FGM_rs                             0.002727
CLOSESTDEF_4_6_FGM_rs_lag2         0.002002
FGM_rs_lag2                        0.001962
FRONT_CT_TOUCHES_rs_lag1           0.001842
PIE_rs_lag1                        0.001618
TOTAL_MIN_rs_lag2                  0.001469
OPP_PTS_OFF_TOV_rs                 0.001449
FRONT_CT_TOUCHES_rs_lag2           0.001423
ALLNBA_PRIOR_EVER                  0.001419
TOTAL_MIN_rs_lag1                  0.001256
FTM_rs_lag2                        0.001134
PIE_rs                             0.001107
POST_TOV_POSS_PCT_rs               0.000801
DRIBB_2_FG_PCT_rs                  0.000796
POST_TOUC

In [137]:
mdf = mdf.copy()
mdf['pred_pct_cap'] = m_lgb.predict(Xall)
mdf['residual']     = mdf['pct_cap'] - mdf['pred_pct_cap']   # +overpaid  /  −underpaid
mdf['resid_usd']    = mdf['residual'] * mdf['SEASON'].map(SALARY_CAP)

cols = ['PLAYER_NAME','SEASON','pct_cap','pred_pct_cap','resid_usd']
print("\n💸 most OVERPAID (model says they earn above their profile):")
print(mdf.sort_values('residual', ascending=False)[cols].head(20).to_string(index=False))
print("\n💎 most UNDERPAID:")
print(mdf.sort_values('residual')[cols].head(20).to_string(index=False))


💸 most OVERPAID (model says they earn above their profile):
          PLAYER_NAME  SEASON  pct_cap  pred_pct_cap    resid_usd
            john wall 2022-23 0.330490      0.090359 2.969338e+07
          ben simmons 2024-25 0.279228      0.089309 2.670027e+07
          ben simmons 2022-23 0.286674      0.108548 2.202622e+07
        fred vanvleet 2024-25 0.304767      0.175165 1.822057e+07
       jonathan isaac 2024-25 0.177825      0.049392 1.805604e+07
        fred vanvleet 2023-24 0.300000      0.172597 1.732944e+07
giannis antetokounmpo 2025-26 0.350000      0.225311 1.928280e+07
           kevin love 2022-23 0.221931      0.099116 1.518659e+07
       michael porter 2023-24 0.245454      0.123853 1.654025e+07
          jalen suggs 2025-26 0.226322      0.107274 1.841040e+07
         myles turner 2022-23 0.283608      0.166059 1.453547e+07
           og anunoby 2025-26 0.255866      0.142713 1.749877e+07
    russell westbrook 2022-23 0.374391      0.261810 1.392117e+07
        jaren j

In [141]:
this_yr = mdf[mdf['SEASON'] == '2025-26'].copy()
print("most overpaid 2025-26:")
print(this_yr.sort_values('residual', ascending=False)[cols].head(60).to_string(index=False))
print("\nmost underpaid 2025-26:")
print(this_yr.sort_values('residual')[cols].head(60).to_string(index=False))

most overpaid 2025-26:
             PLAYER_NAME  SEASON  pct_cap  pred_pct_cap    resid_usd
   giannis antetokounmpo 2025-26 0.350000      0.225311 1.928280e+07
             jalen suggs 2025-26 0.226322      0.107274 1.841040e+07
              og anunoby 2025-26 0.255866      0.142713 1.749877e+07
         khris middleton 2025-26 0.215305      0.105196 1.702799e+07
             zach lavine 2025-26 0.307149      0.202567 1.617335e+07
           anthony davis 2025-26 0.350000      0.263650 1.335374e+07
         zion williamson 2025-26 0.255072      0.170274 1.311371e+07
          darius garland 2025-26 0.255072      0.171653 1.290043e+07
             evan mobley 2025-26 0.300000      0.219750 1.241043e+07
            jordan poole 2025-26 0.205941      0.128502 1.197573e+07
               ja morant 2025-26 0.255072      0.179120 1.174579e+07
             alex caruso 2025-26 0.117054      0.045152 1.111934e+07
                naz reid 2025-26 0.139361      0.075880 9.817081e+06
           

In [96]:
# reuse the explainer + shap values from the global cell (don't recompute)
# sv, expl, Xall, feats, mdf all already exist

def explain_player(name, season, top_n=20):
    row = mdf[(mdf['PLAYER_NAME'] == name) & (mdf['SEASON'] == season)]
    if row.empty:
        print(f"no row for {name} {season}"); return
    i   = mdf.index.get_loc(row.index[0])
    cap = SALARY_CAP[season]

    contrib = pd.Series(sv[i], index=feats)
    top = contrib.reindex(contrib.abs().sort_values(ascending=False).index).head(top_n)

    base = expl.expected_value
    pred = row['pred_pct_cap'].iloc[0]; actual = row['pct_cap'].iloc[0]
    print(f"\n{name} — {season}")
    print(f"  market baseline:  {base:6.1%}  (${base*cap:,.0f})")
    print(f"  model prediction: {pred:6.1%}  (${pred*cap:,.0f})")
    print(f"  actual salary:    {actual:6.1%}  (${actual*cap:,.0f})")
    print(f"  → {'OVER' if actual>pred else 'UNDER'}paid vs market by ${abs(actual-pred)*cap:,.0f}\n")
    print("  what pushed the market price (+ up / − down):")
    for f, v in top.items():
        print(f"    {'+' if v>0 else '−'} {f:32s} {v*cap:>+12,.0f}")

# the players you just flagged
explain_player("giannis antetokounmpo", "2025-26")
explain_player("cj mccollum", "2025-26")
explain_player("julius randle", "2025-26")
explain_player("stephen curry", "2025-26")
explain_player("lebron james", "2025-26")
explain_player("joel embiid", "2025-26")


giannis antetokounmpo — 2025-26
  market baseline:    7.9%  ($12,291,686)
  model prediction:  20.4%  ($31,555,975)
  actual salary:     35.0%  ($54,126,450)
  → OVERpaid vs market by $22,570,475

  what pushed the market price (+ up / − down):
    + EXPERIENCE                         +6,431,776
    + FGM_rs                             +3,571,216
    + ALLNBA_PRIOR_EVER                  +2,306,929
    + FRONT_CT_TOUCHES_rs                +2,240,844
    − MIN_rs                             -1,561,650
    + POST_TOUCHES_rs                    +1,143,260
    + MAX_PCT_ELIGIBLE                   +1,077,006
    + OPP_PTS_OFF_TOV_rs                   +665,455
    + ALLNBA_PRIOR3                        +610,843
    + FTM_rs                               +568,524
    + POST_POSS_rs                         +523,003
    − DRAFT_POSITION                       -513,331
    + DRAFT_YR                             +434,371
    − DRIBB_2_FG_PCT_rs                    -425,867
    + REB_CHANCE_PCT_rs   

In [143]:
ranked = (mdf[mdf['SEASON']=='2025-26']
          .assign(deserved_usd = mdf['pred_pct_cap']*SALARY_CAP['2025-26'])
          .sort_values('pred_pct_cap', ascending=False)
          [['PLAYER_NAME','pred_pct_cap','deserved_usd','pct_cap','resid_usd']])
print(ranked.head(70).to_string(index=False))

            PLAYER_NAME  pred_pct_cap  deserved_usd  pct_cap     resid_usd
           lebron james      0.341513  5.281396e+07 0.340305 -1.868069e+05
          kawhi leonard      0.334501  5.172955e+07 0.323317 -1.729551e+06
           nikola jokic      0.331621  5.128417e+07 0.357101  3.940354e+06
           kevin durant      0.328532  5.080656e+07 0.353764  3.902050e+06
          stephen curry      0.322415  4.986055e+07 0.385438  9.746262e+06
            luka doncic      0.317096  4.903801e+07 0.297449 -3.038355e+06
           james harden      0.311793  4.821782e+07 0.253369 -9.035124e+06
            joel embiid      0.310806  4.806518e+07 0.357101  7.159344e+06
           jaylen brown      0.301026  4.655269e+07 0.343636  6.589571e+06
           devin booker      0.297574  4.601889e+07 0.343636  7.123377e+06
          julius randle      0.293032  4.531658e+07 0.199578 -1.445238e+07
       donovan mitchell      0.290166  4.487331e+07 0.300000  1.520792e+06
     karl-anthony towns  

# Save Model

In [126]:
# clean model (idempotent guard for any future lag re-runs)
model = model.loc[:, ~model.columns.duplicated()]

# 1) the trained model — both formats
m_lgb.booster_.save_model("data/salary_model_v1.txt")
joblib.dump(m_lgb, "data/salary_model_v1.pkl")

# 2) the exact feature list + order (model is useless without this)
json.dump(feats, open("data/feats_v1.json", "w"))

# 3) the residual table — the product, queryable
out_cols = ['PLAYER_NAME','SEASON','pct_cap','pred_pct_cap','resid_usd']
mdf[out_cols].to_parquet("data/residuals_v1.parquet")

# 4) the scoreable population (everyone, not just training rows) for the app
mdf.to_parquet("data/mdf_v1.parquet")

print("saved:", report)   # log the prefilter summary so you know what built this
print(f"R2 0.847 | MAE $3.75M | {len(feats)} features | {len(mdf)} rows")

saved: {'start': 1487, 'const': 1, 'empty': 2, 'collinear': 449, 'keep': 1035}
R2 0.847 | MAE $3.75M | 1037 features | 4228 rows


# Streamlit App

In [127]:
# global ranking → the features worth storing for the app
glob = pd.Series(np.abs(sv).mean(0), index=feats).sort_values(ascending=False)
DISPLAY_FEATS = glob.head(25).index.tolist()          # the app shows top contributors per player
display_idx   = [feats.index(f) for f in DISPLAY_FEATS]

# build a long-format SHAP table: one row per (player-season, feature) = easy for plotly
rows = []
cap_by_season = mdf['SEASON'].map(SALARY_CAP).values
for r in range(len(mdf)):
    cap = cap_by_season[r]
    for f, j in zip(DISPLAY_FEATS, display_idx):
        rows.append((mdf['PLAYER_NAME'].iloc[r], mdf['SEASON'].iloc[r],
                     f, sv[r, j], sv[r, j] * cap, mdf[f].iloc[r]))

shap_long = pd.DataFrame(rows, columns=['PLAYER_NAME','SEASON','feature',
                                        'shap','shap_usd','feat_value'])
shap_long.to_parquet("data/shap_v1.parquet")

# also store the per-player baseline + prediction so the waterfall has its anchors
base = float(expl.expected_value)
meta = mdf[['PLAYER_NAME','SEASON','pct_cap','pred_pct_cap','resid_usd']].copy()
meta['baseline_pct'] = base
meta.to_parquet("data/app_meta_v1.parquet")

print(f"stored SHAP for {len(DISPLAY_FEATS)} features × {len(mdf)} player-seasons "
      f"= {len(shap_long):,} rows ({shap_long.memory_usage(deep=True).sum()/1e6:.1f} MB)")
print("baseline pct_cap:", round(base, 4))

stored SHAP for 25 features × 4228 player-seasons = 105,700 rows (24.2 MB)
baseline pct_cap: 0.0796


In [129]:
pip install -r data/requirements.txt

94527.50s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 37.0 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.2/731.2 kB 13.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 49.1 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 41.7 MB/s  0:00:00 eta 0:00:01
  Attempting uninstall: packaging0m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  4/20 [protobuf]
    Found existing installation: packaging 26.2━━━━━━━━━━━━━━━  4/20 [protobuf]
    Uninstalling packaging-26.2:━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  4/20 [protobuf]
      Successfully uninstalled packaging-26.2━━━━━━━━━━━━━━━━━  4/20 [protobuf]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20/20 [streamlit]20 [streamlit]]
Note: you may need to restart the kernel to use updated packages.
